# Kiểm tra loading và split dữ liệu MELD

Notebook này kiểm tra nhanh cấu trúc `data/MELD/`, schema CSV, phân bố split/label, khả năng resolve audio trong các shard, và thử tạo dataset/dataloader tối giản cho train/valid/test.

Mặc định notebook chỉ đọc dữ liệu. Cell cuối có tùy chọn xuất manifest đã resolve audio path sang `data/splits/meld_*.csv` nếu cần dùng trực tiếp cho training speech/fusion.

In [1]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import random
import wave

import pandas as pd
from IPython.display import display


## 1. Cấu hình đường dẫn

In [2]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / 'data' / 'MELD'
AUDIO_ROOT = DATA_ROOT / 'original_meld_audio_wav'

SPLITS = ['train', 'valid', 'test']
LABEL2ID = {
    'anger': 0,
    'fear': 1,
    'happiness': 2,
    'sadness': 3,
    'neutral': 4,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATA_ROOT    =', DATA_ROOT)
print('AUDIO_ROOT   =', AUDIO_ROOT)

assert DATA_ROOT.exists(), f'Missing data root: {DATA_ROOT}'
assert AUDIO_ROOT.exists(), f'Missing audio root: {AUDIO_ROOT}'


PROJECT_ROOT = /home/emotalk/mer2
DATA_ROOT    = /home/emotalk/mer2/data/MELD
AUDIO_ROOT   = /home/emotalk/mer2/data/MELD/original_meld_audio_wav


## 2. Đọc kiến trúc thư mục

In [3]:
top_level = []
for p in sorted(DATA_ROOT.iterdir(), key=lambda x: x.name):
    top_level.append({
        'name': p.name,
        'type': 'dir' if p.is_dir() else 'file',
        'size_bytes': p.stat().st_size if p.is_file() else None,
    })

display(pd.DataFrame(top_level))


,name,type,size_bytes
0,extract_manifest.csv,file,8887148.0
1,label_stats.csv,file,279.0
2,meld_vi_text_original_audio_all.csv,file,8901891.0
3,meld_vi_text_original_audio_bad.csv,file,112761.0
4,meld_vi_text_original_audio_clean.csv,file,8218030.0
5,original_meld_audio_wav,dir,NaN
6,quality_summary.csv,file,201.0
7,test.csv,file,1619929.0
8,train.csv,file,5937005.0
9,valid.csv,file,661520.0


In [4]:
shard_rows = []
for shard in sorted(AUDIO_ROOT.glob('original_meld_audio_wav_shard_*')):
    if shard.is_dir():
        shard_rows.append({
            'shard': shard.name,
            'wav_files': len(list(shard.rglob('*.wav'))),
        })

shard_df = pd.DataFrame(shard_rows)
display(shard_df)
print('Total wav:', int(shard_df['wav_files'].sum()) if not shard_df.empty else 0)


,shard,wav_files
0,original_meld_audio_wav_shard_0000,1000
1,original_meld_audio_wav_shard_0001,1000
2,original_meld_audio_wav_shard_0002,1000
3,original_meld_audio_wav_shard_0003,1000
4,original_meld_audio_wav_shard_0004,1000
5,original_meld_audio_wav_shard_0005,1000
6,original_meld_audio_wav_shard_0006,1000
7,original_meld_audio_wav_shard_0007,1000
8,original_meld_audio_wav_shard_0008,1000
9,original_meld_audio_wav_shard_0009,1000


Total wav: 11686


## 3. Đọc CSV và kiểm tra schema

In [5]:
def read_csv(name: str) -> pd.DataFrame:
    path = DATA_ROOT / name
    assert path.exists(), f'Missing CSV: {path}'
    return pd.read_csv(path)

frames = {split: read_csv(f'{split}.csv') for split in SPLITS}
frames['clean'] = read_csv('meld_vi_text_original_audio_clean.csv')
frames['all'] = read_csv('meld_vi_text_original_audio_all.csv')
frames['bad'] = read_csv('meld_vi_text_original_audio_bad.csv')

summary_rows = []
for name, df in frames.items():
    summary_rows.append({
        'frame': name,
        'rows': len(df),
        'columns': len(df.columns),
        'column_names': ', '.join(df.columns),
    })

display(pd.DataFrame(summary_rows))


,frame,rows,columns,column_names
0,train,8354,19,"sample_id, utterance_id, split, label, text, r..."
1,valid,919,19,"sample_id, utterance_id, split, label, text, r..."
2,test,2211,19,"sample_id, utterance_id, split, label, text, r..."
3,clean,11484,19,"sample_id, utterance_id, split, label, text, r..."
4,all,11687,29,"sample_id, utterance_id, split, label, text, r..."
5,bad,203,29,"sample_id, utterance_id, split, label, text, r..."


In [6]:
required_columns = {
    'sample_id',
    'split',
    'label',
    'text',
    'transcript_final',
    'audio_path',
    'audio_relpath',
    'group_id',
    'duration_sec',
    'sample_rate',
}

schema_checks = []
for split, df in frames.items():
    if split not in SPLITS:
        continue
    missing = sorted(required_columns - set(df.columns))
    schema_checks.append({
        'split': split,
        'missing_required_columns': ', '.join(missing),
        'ok': len(missing) == 0,
    })

display(pd.DataFrame(schema_checks))
display(frames['train'].head(3).T)


,split,missing_required_columns,ok
0,train,,True
1,valid,,True
2,test,,True


,0,1,2
sample_id,meld_train_d0000_u000,meld_train_d0000_u001,meld_train_d0000_u002
utterance_id,meld_train_d0000_u000,meld_train_d0000_u001,meld_train_d0000_u002
split,train,train,train
label,neutral,neutral,neutral
text,Ngoài ra tôi là người quan trọng trong quá trì...,Chắc tay anh đầy rồi.,"Đúng vậy, đúng vậy."
raw_text,Ngoài ra tôi là người quan trọng trong quá trì...,Chắc tay anh đầy rồi.,"Đúng vậy, đúng vậy."
normalized_text,Ngoài ra tôi là người quan trọng trong quá trì...,Chắc tay anh đầy rồi.,"Đúng vậy, đúng vậy."
transcript_final,Ngoài ra tôi là người quan trọng trong quá trì...,Chắc tay anh đầy rồi.,"Đúng vậy, đúng vậy."
text_for_model,Ngoài ra tôi là người quan trọng trong quá trì...,Chắc tay anh đầy rồi.,"Đúng vậy, đúng vậy."
audio_path,original_meld_audio_wav/meld_train_d0000_u000.wav,original_meld_audio_wav/meld_train_d0000_u001.wav,original_meld_audio_wav/meld_train_d0000_u002.wav


## 4. Kiểm tra split và phân bố nhãn

In [7]:
split_rows = []
for split, df in frames.items():
    if split not in SPLITS:
        continue
    split_rows.append({
        'split_file': split,
        'rows': len(df),
        'declared_split_values': ', '.join(sorted(df['split'].astype(str).unique())),
        'dialogues': df['group_id'].nunique(),
        'speakers': df['speaker'].nunique() if 'speaker' in df.columns else None,
    })

display(pd.DataFrame(split_rows))

combined = pd.concat([frames[s].assign(split_file=s) for s in SPLITS], ignore_index=True)
label_table = pd.crosstab(combined['split_file'], combined['label'])
display(label_table[list(LABEL2ID.keys())])

unknown_labels = sorted(set(combined['label']) - set(LABEL2ID))
print('Unknown labels:', unknown_labels)
assert not unknown_labels, f'Unknown labels found: {unknown_labels}'


,split_file,rows,declared_split_values,dialogues,speakers
0,train,8354,train,1024,249
1,valid,919,valid,114,46
2,test,2211,test,278,98


label,anger,fear,happiness,sadness,neutral
split_file,,,,,
test,332,50,394,205,1230
train,1090,265,1704,676,4619
valid,149,39,162,109,460


Unknown labels: []


In [8]:
id_sets = {split: set(frames[split]['sample_id'].astype(str)) for split in SPLITS}
group_sets = {split: set(frames[split]['group_id'].astype(str)) for split in SPLITS}

overlap_rows = []
for i, left in enumerate(SPLITS):
    for right in SPLITS[i + 1:]:
        overlap_rows.append({
            'pair': f'{left} vs {right}',
            'sample_id_overlap': len(id_sets[left] & id_sets[right]),
            'group_id_overlap': len(group_sets[left] & group_sets[right]),
        })

display(pd.DataFrame(overlap_rows))

dup_rows = []
for split in SPLITS:
    df = frames[split]
    dup_rows.append({
        'split': split,
        'duplicate_sample_id_rows': int(df['sample_id'].duplicated().sum()),
        'empty_text_rows': int(df['text'].fillna('').astype(str).str.strip().eq('').sum()),
        'empty_audio_path_rows': int(df['audio_path'].fillna('').astype(str).str.strip().eq('').sum()),
    })

display(pd.DataFrame(dup_rows))


,pair,sample_id_overlap,group_id_overlap
0,train vs valid,0,0
1,train vs test,0,0
2,valid vs test,0,0


,split,duplicate_sample_id_rows,empty_text_rows,empty_audio_path_rows
0,train,0,0,0
1,valid,0,0,0
2,test,0,0,0


## 5. Kiểm tra audio path trong shard

In [9]:
audio_paths = sorted(AUDIO_ROOT.rglob('*.wav'))
audio_index = {}
duplicate_audio_names = []

for path in audio_paths:
    if path.name in audio_index:
        duplicate_audio_names.append(path.name)
    audio_index[path.name] = path

print('Indexed wav files:', len(audio_index))
print('Duplicate audio basenames:', len(duplicate_audio_names))
assert len(duplicate_audio_names) == 0, 'Duplicate audio basenames found; basename resolution is unsafe.'

display(pd.DataFrame({'example_audio_path': [str(p.relative_to(PROJECT_ROOT)) for p in audio_paths[:5]]}))


Indexed wav files: 11686
Duplicate audio basenames: 0


,example_audio_path
0,data/MELD/original_meld_audio_wav/original_mel...
1,data/MELD/original_meld_audio_wav/original_mel...
2,data/MELD/original_meld_audio_wav/original_mel...
3,data/MELD/original_meld_audio_wav/original_mel...
4,data/MELD/original_meld_audio_wav/original_mel...


In [10]:
all_meld_wavs = sorted(DATA_ROOT.rglob('*.wav'))
audio_root_wavs = sorted(AUDIO_ROOT.rglob('*.wav'))
outside_audio_root = [
    p for p in all_meld_wavs
    if not p.resolve().is_relative_to(AUDIO_ROOT.resolve())
]

quality_summary_wav_files = None
quality_summary_path = DATA_ROOT / 'quality_summary.csv'
if quality_summary_path.exists():
    quality_summary_df = pd.read_csv(quality_summary_path)
    wav_file_rows = quality_summary_df.loc[
        quality_summary_df['metric'].astype(str).eq('wav_files'),
        'value',
    ]
    if not wav_file_rows.empty:
        quality_summary_wav_files = int(wav_file_rows.iloc[0])

wav_count_check = pd.DataFrame([
    {'source': 'AUDIO_ROOT recursive *.wav', 'count': len(audio_root_wavs)},
    {'source': 'DATA_ROOT/MELD recursive *.wav', 'count': len(all_meld_wavs)},
    {'source': 'unique wav basenames in MELD', 'count': len({p.name for p in all_meld_wavs})},
    {'source': 'quality_summary.csv wav_files', 'count': quality_summary_wav_files},
])
display(wav_count_check)

print('Wav files outside AUDIO_ROOT:', len(outside_audio_root))
if outside_audio_root:
    display(pd.DataFrame({'outside_audio_root': [str(p.relative_to(PROJECT_ROOT)) for p in outside_audio_root[:20]]}))

assert len(audio_root_wavs) == len(all_meld_wavs), 'AUDIO_ROOT wav count differs from total MELD wav count.'
assert len({p.name for p in all_meld_wavs}) == len(all_meld_wavs), 'Duplicate wav basenames found in MELD.'
if quality_summary_wav_files is not None:
    assert len(all_meld_wavs) == quality_summary_wav_files, 'MELD wav count differs from quality_summary.csv wav_files.'


,source,count
0,AUDIO_ROOT recursive *.wav,11686
1,DATA_ROOT/MELD recursive *.wav,11686
2,unique wav basenames in MELD,11686
3,quality_summary.csv wav_files,11686


Wav files outside AUDIO_ROOT: 0


In [11]:
def resolve_audio_from_row(row: pd.Series) -> Path | None:
    raw_path = str(row.get('audio_path') or row.get('audio_relpath') or '').strip()
    if not raw_path:
        return None

    candidates = [
        PROJECT_ROOT / raw_path,
        DATA_ROOT / raw_path,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    return audio_index.get(Path(raw_path).name)


resolved_frames = {}
path_check_rows = []

for split in SPLITS:
    df = frames[split].copy()
    resolved = df.apply(resolve_audio_from_row, axis=1)
    df['resolved_audio_path'] = resolved.map(lambda p: str(p) if p is not None else '')
    df['audio_exists'] = df['resolved_audio_path'].astype(str).str.len().gt(0)
    df['label_id'] = df['label'].map(LABEL2ID).astype('int64')
    df['duration'] = df['duration_sec'].astype('float64')
    df['audio_path_for_training'] = df['resolved_audio_path'].map(
        lambda x: Path(x).relative_to(PROJECT_ROOT).as_posix() if str(x) else ''
    )
    resolved_frames[split] = df

    path_check_rows.append({
        'split': split,
        'rows': len(df),
        'audio_resolved': int(df['audio_exists'].sum()),
        'missing_audio': int((~df['audio_exists']).sum()),
        'direct_project_path_exists': int(df['audio_path'].map(lambda x: (PROJECT_ROOT / str(x)).exists()).sum()),
        'direct_data_root_path_exists': int(df['audio_path'].map(lambda x: (DATA_ROOT / str(x)).exists()).sum()),
    })

display(pd.DataFrame(path_check_rows))

missing_audio = pd.concat(
    [df.loc[~df['audio_exists'], ['sample_id', 'split', 'audio_path']] for df in resolved_frames.values()],
    ignore_index=True,
)
display(missing_audio.head(20))
assert missing_audio.empty, f'Missing audio rows: {len(missing_audio)}'


,split,rows,audio_resolved,missing_audio,direct_project_path_exists,direct_data_root_path_exists
0,train,8354,8354,0,0,0
1,valid,919,919,0,0,0
2,test,2211,2211,0,0,0


,sample_id,split,audio_path


## 6. Kiểm tra metadata WAV và thử load một mẫu

In [12]:
def wav_metadata(path: str | Path) -> dict[str, object]:
    path = Path(path)
    with wave.open(str(path), 'rb') as wf:
        frames = wf.getnframes()
        sample_rate = wf.getframerate()
        channels = wf.getnchannels()
        sample_width = wf.getsampwidth()
    return {
        'path': str(path.relative_to(PROJECT_ROOT)),
        'channels': channels,
        'sample_rate': sample_rate,
        'sample_width_bytes': sample_width,
        'frames': frames,
        'duration_sec_from_wav': frames / sample_rate if sample_rate else None,
    }


sample_rows = []
for split in SPLITS:
    df = resolved_frames[split]
    row = df.sample(1, random_state=42).iloc[0]
    meta = wav_metadata(row['resolved_audio_path'])
    meta.update({
        'split': split,
        'sample_id': row['sample_id'],
        'label': row['label'],
        'duration_sec_csv': float(row['duration_sec']),
    })
    sample_rows.append(meta)

display(pd.DataFrame(sample_rows))


,path,channels,sample_rate,sample_width_bytes,frames,duration_sec_from_wav,split,sample_id,label,duration_sec_csv
0,data/MELD/original_meld_audio_wav/original_mel...,1,24000,2,104448,4.352000,train,meld_train_d0457_u004,neutral,4.352000
1,data/MELD/original_meld_audio_wav/original_mel...,1,24000,2,82432,3.434667,valid,meld_valid_d0009_u002,neutral,3.434667
2,data/MELD/original_meld_audio_wav/original_mel...,1,24000,2,69120,2.880000,test,meld_test_d0112_u005,neutral,2.880000


In [13]:
def load_audio_preview(path: str | Path):
    path = str(path)
    try:
        import soundfile as sf
        audio, sr = sf.read(path, always_2d=True)
        return audio, sr, 'soundfile'
    except Exception as soundfile_error:
        try:
            import torchaudio
            waveform, sr = torchaudio.load(path)
            return waveform, sr, 'torchaudio'
        except Exception as torchaudio_error:
            import numpy as np
            with wave.open(path, 'rb') as wf:
                sr = wf.getframerate()
                channels = wf.getnchannels()
                sample_width = wf.getsampwidth()
                raw = wf.readframes(wf.getnframes())
            if sample_width == 1:
                audio = np.frombuffer(raw, dtype=np.uint8).astype('float32') - 128.0
            elif sample_width == 2:
                audio = np.frombuffer(raw, dtype=np.int16).astype('float32')
            elif sample_width == 4:
                audio = np.frombuffer(raw, dtype=np.int32).astype('float32')
            else:
                raise RuntimeError(
                    f'Cannot load audio. soundfile={soundfile_error}; '
                    f'torchaudio={torchaudio_error}; unsupported sample_width={sample_width}'
                )
            audio = audio.reshape(-1, channels)
            return audio, sr, 'wave+numpy'


example = resolved_frames['train'].iloc[0]
audio, sr, backend = load_audio_preview(example['resolved_audio_path'])
print('backend:', backend)
print('sample_id:', example['sample_id'])
print('label:', example['label'], 'label_id:', example['label_id'])
print('text:', example['text'])
print('audio shape:', getattr(audio, 'shape', None))
print('sample_rate:', sr)


backend: soundfile
sample_id: meld_train_d0000_u000
label: neutral label_id: 4
text: Ngoài ra tôi là người quan trọng trong quá trình chuyển đổi của công ty tôi từ hệ thống KL-5 sang GR-6.
audio shape: (136192, 1)
sample_rate: 24000


## 7. Dataset/Dataloader tối giản để kiểm tra training input

In [14]:
try:
    import torch
    from torch.utils.data import DataLoader, Dataset
    TORCH_AVAILABLE = True
except Exception as exc:
    TORCH_AVAILABLE = False
    print('Torch unavailable:', exc)


if TORCH_AVAILABLE:
    class MELDResolvedDataset(Dataset):
        def __init__(self, frame: pd.DataFrame):
            self.frame = frame.reset_index(drop=True)

        def __len__(self) -> int:
            return len(self.frame)

        def __getitem__(self, idx: int) -> dict[str, object]:
            row = self.frame.iloc[idx]
            return {
                'id': str(row['sample_id']),
                'text': str(row['text']),
                'raw_text': str(row['transcript_final']),
                'audio_path': str(row['resolved_audio_path']),
                'label': str(row['label']),
                'label_id': int(row['label_id']),
                'group_id': str(row['group_id']),
                'duration': float(row['duration']),
                'split': str(row['split']),
            }

    def collate_metadata(batch: list[dict[str, object]]) -> dict[str, object]:
        return {
            'ids': [item['id'] for item in batch],
            'texts': [item['text'] for item in batch],
            'audio_paths': [item['audio_path'] for item in batch],
            'labels': torch.tensor([item['label_id'] for item in batch], dtype=torch.long),
            'group_ids': [item['group_id'] for item in batch],
            'durations': torch.tensor([item['duration'] for item in batch], dtype=torch.float32),
        }

    datasets = {split: MELDResolvedDataset(resolved_frames[split]) for split in SPLITS}
    loaders = {
        split: DataLoader(ds, batch_size=4, shuffle=(split == 'train'), collate_fn=collate_metadata)
        for split, ds in datasets.items()
    }

    batch = next(iter(loaders['train']))
    print('Dataset sizes:', {split: len(ds) for split, ds in datasets.items()})
    print('Batch keys:', list(batch.keys()))
    print('Batch ids:', batch['ids'])
    print('Batch labels:', batch['labels'].tolist())
    print('First audio path exists:', Path(batch['audio_paths'][0]).exists())
    print('First text:', batch['texts'][0])


Dataset sizes: {'train': 8354, 'valid': 919, 'test': 2211}
Batch keys: ['ids', 'texts', 'audio_paths', 'labels', 'group_ids', 'durations']
Batch ids: ['meld_train_d0031_u001', 'meld_train_d0623_u003', 'meld_train_d0329_u010', 'meld_train_d0649_u020']
Batch labels: [4, 0, 4, 4]
First audio path exists: True
First text: Vậy thì nó vẫn chưa có trên tường.


## 8. Tùy chọn xuất manifest đã resolve audio path

Bật `EXPORT_RESOLVED_MANIFESTS = True` nếu muốn tạo CSV dùng trực tiếp cho config training. File sẽ được ghi vào `data/splits/meld_train.csv`, `meld_valid.csv`, `meld_test.csv`.

In [15]:
EXPORT_RESOLVED_MANIFESTS = False
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'splits'

manifest_columns = [
    'sample_id',
    'label',
    'label_id',
    'text',
    'transcript_final',
    'audio_path_for_training',
    'group_id',
    'duration',
    'split',
    'speaker',
    'text_en',
    'text_vi',
]

if EXPORT_RESOLVED_MANIFESTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for split in SPLITS:
        out = resolved_frames[split][manifest_columns].rename(
            columns={'audio_path_for_training': 'audio_path'}
        )
        output_path = OUTPUT_DIR / f'meld_{split}.csv'
        out.to_csv(output_path, index=False, encoding='utf-8')
        print('Wrote', output_path, 'rows=', len(out))
else:
    print('EXPORT_RESOLVED_MANIFESTS is False, no files written.')


EXPORT_RESOLVED_MANIFESTS is False, no files written.


## 9. Kết luận kiểm tra

In [16]:
final_checks = {
    'train_rows': len(resolved_frames['train']),
    'valid_rows': len(resolved_frames['valid']),
    'test_rows': len(resolved_frames['test']),
    'total_clean_rows': sum(len(resolved_frames[s]) for s in SPLITS),
    'audio_files_indexed': len(audio_index),
    'all_audio_resolved': all(resolved_frames[s]['audio_exists'].all() for s in SPLITS),
    'no_unknown_labels': len(unknown_labels) == 0,
    'no_sample_id_overlap': all((id_sets[left] & id_sets[right]) == set() for i, left in enumerate(SPLITS) for right in SPLITS[i + 1:]),
}

display(pd.DataFrame([final_checks]).T.rename(columns={0: 'value'}))
assert final_checks['all_audio_resolved']
assert final_checks['no_unknown_labels']
assert final_checks['no_sample_id_overlap']
print('OK: MELD loading and split checks passed.')


,value
train_rows,8354
valid_rows,919
test_rows,2211
total_clean_rows,11484
audio_files_indexed,11686
all_audio_resolved,True
no_unknown_labels,True
no_sample_id_overlap,True


OK: MELD loading and split checks passed.
